In [1]:
import pandas as pd

In [2]:
df_log = pd.read_csv('data/full_event_log.csv', parse_dates=['timestamp', 'timestamp_end'])

In [13]:
df_case = df_log[df_log['case_id'] == 'CA100']
df_case = df_case.reset_index().rename(columns={'index': 'id'})
df_case

,id,case_id,activity,timestamp,timestamp_end
0,0,CA100,Traslado a carga,2024-02-27 14:01:02,2024-02-27 14:05:56
1,1,CA100,Carga,2024-02-27 14:05:56,2024-02-27 14:07:51
2,2,CA100,Traslado a descarga,2024-02-27 14:07:51,2024-02-27 14:29:21
3,3,CA100,Descarga,2024-02-27 14:29:21,2024-02-27 14:30:14
4,4,CA100,Traslado a carga,2024-02-27 14:30:14,2024-02-27 14:56:13
...,...,...,...,...,...
5049,5049,CA100,Traslado a descarga,2024-03-26 07:50:16,2024-03-26 08:16:23
5050,5050,CA100,Carga de combustible,2024-03-26 08:07:54,2024-03-26 08:16:45
5051,5051,CA100,Traslado a descarga,2024-03-26 08:16:23,2024-03-26 08:16:43
5052,5052,CA100,Descarga,2024-03-26 08:16:43,2024-03-26 08:17:34


In [24]:
df_case_lagged = df_case[['id', 'activity', 'timestamp', 'timestamp_end']].shift(-1).dropna()
df_case_lagged['id'] = df_case['id'].astype(int)
df_case_lagged = df_case_lagged.rename(columns={'activity': 'future_activity', 'timestamp': 'future_timestamp', 'timestamp_end': 'future_timestamp_end'})
df_join = pd.merge(df_case, df_case_lagged, on='id').dropna()

In [33]:
# Actividad que comienza después de que la anterior termine
case_total_events = df_case.shape[0]
overlapping_events = df_join[df_join['timestamp_end'] > df_join['future_timestamp']][['activity', 'future_activity']].shape[0]
print(f'{overlapping_events / case_total_events:.2%}')

21.15%


- Esto dice que en un ~21% de las actividades en este caso (`case_id` como `truck_id`) la actividad siguiente comienza, mientras no termina la anterior, esto produce paralelismo de las actividades

In [34]:
# See what activities are contained completed
df_contained_events = df_join[(df_join['timestamp'] < df_join['future_timestamp']) &\
                              (df_join['timestamp_end'] > df_join['future_timestamp_end'])
                             ]
df_contained_events

,id,case_id,activity,timestamp,timestamp_end,future_activity,future_timestamp,future_timestamp_end
13,13,CA100,Pista obstruida,2024-02-27 15:30:40,2024-02-27 15:34:55,Traslado a carga,2024-02-27 15:32:25,2024-02-27 15:34:25
20,20,CA100,Traslado a descarga,2024-02-27 15:49:42,2024-02-27 16:05:44,Sin equipo de carguio,2024-02-27 16:01:19,2024-02-27 16:04:09
25,25,CA100,Colacion comedor,2024-02-27 16:13:36,2024-02-27 16:34:17,Traslado a carga,2024-02-27 16:20:54,2024-02-27 16:22:46
39,39,CA100,Traslado a descarga,2024-02-27 17:38:23,2024-02-27 17:53:56,Carga de combustible,2024-02-27 17:42:34,2024-02-27 17:53:10
78,78,CA100,Cambio de turno,2024-02-27 23:49:31,2024-02-28 00:00:00,Descarga,2024-02-27 23:52:49,2024-02-27 23:52:51
...,...,...,...,...,...,...,...,...
5024,5024,CA100,Traslado a carga,2024-03-26 03:05:51,2024-03-26 03:48:28,Colacion cabina,2024-03-26 03:20:28,2024-03-26 03:41:02
5028,5028,CA100,Traslado a descarga,2024-03-26 03:50:19,2024-03-26 04:16:21,Pista obstruida,2024-03-26 04:01:42,2024-03-26 04:04:25
5038,5038,CA100,Operador fuera del equipo,2024-03-26 06:10:41,2024-03-26 06:40:44,Traslado a descarga,2024-03-26 06:13:43,2024-03-26 06:38:49
5044,5044,CA100,Colacion cabina,2024-03-26 07:21:32,2024-03-26 08:02:08,Traslado a descarga,2024-03-26 07:23:45,2024-03-26 07:24:04
